# Quick benchmark (offline)
Compare LCDM, SAC, and a LOOP-style planner on a DMControl task (pixels).

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

import torch

import matplotlib.pyplot as plt
import pandas as pd


from lcdm import LCDM
from sac import SACAgent
from loop import LOOPAgent, LOOPPlanner
from env import DMCEnv


device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# create a pixel-based cartpole swingup environment (change domain/task as desired)
env = DMCEnv(domain_name="cartpole", task_name="swingup", from_pixels=True, img_size=64)
obs = env.reset()


In [ ]:

action_spec = env.get_actions()
try:
    action_shape = action_spec.shape
except Exception:
    try:
        action_shape = action_spec.tensor_spec.shape
    except Exception:
        action_shape = (1,)
action_dim = int(action_shape[0]) if len(action_shape) > 0 else 1

# instantiate agents (encoder expects CHW images)
lcdm = LCDM(obs_shape=tuple(obs.shape), action_dim=action_dim, img_size=64, latent_dim=16, device=device)
sac = SACAgent(obs_shape=tuple(obs.shape), action_dim=action_dim, device=device, use_encoder=True, encoder_params={'in_channels':3,'num_channels':32,'img_size':64,'latent_dim':16})
loop_agent = LOOPAgent(latent_dim=16, action_dim=action_dim, device=device)
loop_planner = LOOPPlanner(latent_dim=16, action_dim=action_dim, horizon=5, num_samples=128, sigma=0.3, device=device)

In [ ]:
# experiment settings
episodes = 100
steps_per_ep = 200
results = {'lcdm': [], 'sac': [], 'loop': []}
print(f"Running {episodes} episodes x {steps_per_ep} steps per agent; action_dim={action_dim}")


In [ ]:
for alg in ['lcdm', 'sac', 'loop']:
    for ep in range(episodes):
        obs = env.reset()
        ep_return = 0.0
        for t in range(steps_per_ep):
            if alg == 'lcdm':
                a = lcdm.act(obs)
            elif alg == 'sac':
                a = sac.act(obs)
            else:
                with torch.no_grad():
                    z = lcdm.encode(obs.unsqueeze(0))
                # pass the current causal graph adjacency to the planner 
                # so the transition model receives it
                a = loop_planner.plan(
                    lcdm.transition_model, 
                    lcdm.reward_model, 
                    loop_agent.value_net, 
                    z, 
                    lcdm.causal_graph()
                )
            next_obs, reward, done, _ = env.step(a)
            ep_return += float(reward)
            obs = next_obs
        results[alg].append(ep_return)
        print(f"{alg} ep {ep+1} return={ep_return:.3f}")


plt.figure(figsize=(6,4))
ax = plt.gca()
ax.boxplot([results['lcdm'], results['sac'], results['loop']], labels=['lcdm','sac','loop'])
ax.set_title('Quick DMC benchmark: returns')
plt.show()

df = pd.DataFrame({k:pd.Series(v) for k,v in results.items()})
display(df)
